In [ ]:
"""
XGBoost 피처 매트릭스 조립 v3 - v2(정규 격자 + as-of join) + is_bottleneck_slot
leakage 제거

v2까지는 is_bottleneck_slot이 전체 639일 데이터로 계산되어 있어 Train
기간 학습 시 미래(Val/Test 기간) 정보가 섞이는 leakage 위험이 있었다.
v3는 bottleneck_train_only.ipynb에서 Train 기간(<2026-06-14)만으로 다시
계산한 is_bottleneck_slot_TRAIN_ONLY.csv를 사용한다. 그 외 로직은 v2와
완전히 동일(정규 10분 격자 + as-of join).

참고: 전체기간 버전과 비교했을 때 판정이 달라진 슬롯은 90개 구간 x 48
슬롯 중 9개(0.2%)뿐이었다 - leakage가 실재하지만 정의 자체에 미치는 영향은
작았다. 다만 모델 성능 수치의 신뢰도를 위해 정확한 버전으로 교체한다.

출력: output/features/xgb_feature_matrix_v3.parquet
"""

from pathlib import Path

import polars as pl

FEATURES_DIR = Path("./output/features")
EDA_DIR = Path("./output/eda")

SPEED_PATH = FEATURES_DIR / "speed_features.parquet"
BOTTLENECK_PATH = EDA_DIR / "is_bottleneck_slot_TRAIN_ONLY.csv"  # v2에서 바뀐 부분
NETWORK_PATH = FEATURES_DIR / "network_features.parquet"
LANE_RATIO_PATH = FEATURES_DIR / "construction_lane_ratio_daily.parquet"
INCIDENT_PATH = FEATURES_DIR / "incident_flag_10min.parquet"
WEATHER_PATH = FEATURES_DIR / "weather_features.parquet"
PROPHET_PATH = FEATURES_DIR / "prophet_features.parquet"
SEGMENT_LINK_MAPPING_PATH = "./output/segment_link_mapping.csv"

OUTPUT_PATH = FEATURES_DIR / "xgb_feature_matrix_v3.parquet"  # v2에서 바뀐 부분

TS_UNIT = "us"
GRID_INTERVAL = "10m"
BACKWARD_TOLERANCE = "30m"
LABEL_TOLERANCE = "10m"
TRAIN_END = "2026-06-14"
VAL_END = "2026-06-28"

print(f"grid={GRID_INTERVAL}, backward_tol={BACKWARD_TOLERANCE}, label_tol={LABEL_TOLERANCE}")
print(f"bottleneck source: {BOTTLENECK_PATH}")

In [ ]:
# ==================================================================
# 1. 정규 격자(canonical grid) 생성: 90개 구간 x 10분 간격 cross join
# ==================================================================

seg_map = pl.read_csv(SEGMENT_LINK_MAPPING_PATH)
segment_keys = (seg_map["segment_id"] + "_" + seg_map["direction"]).unique().sort()
print(f"segment_key 수: {len(segment_keys)}")

speed = pl.read_parquet(SPEED_PATH).with_columns(pl.col("timestamp").cast(pl.Datetime(TS_UNIT)))
ts_min, ts_max = speed["timestamp"].min(), speed["timestamp"].max()
print(f"기간: {ts_min} ~ {ts_max}")

grid_times = pl.datetime_range(ts_min, ts_max, interval=GRID_INTERVAL, eager=True).cast(pl.Datetime(TS_UNIT))
print(f"격자 시각 수: {len(grid_times)}")

grid = pl.DataFrame({"segment_key": segment_keys}).join(pl.DataFrame({"timestamp": grid_times}), how="cross")
print(f"정규 격자 shape: {grid.shape}")

In [ ]:
# ==================================================================
# 2. 현재상태 속도 피처 as-of join (backward, 최대 30분 이내)
# ==================================================================

speed_sorted = speed.sort(["segment_key", "timestamp"])
grid_sorted = grid.sort(["segment_key", "timestamp"])

df = grid_sorted.join_asof(
    speed_sorted,
    on="timestamp",
    by="segment_key",
    strategy="backward",
    tolerance=BACKWARD_TOLERANCE,
)

print(f"shape: {df.shape}")
print(f"V_segment 결측률: {df['V_segment'].null_count() / df.height:.2%}")

In [ ]:
# ==================================================================
# 3. 시간 파생 피처 + 조인용 보조 키
# ==================================================================

df = df.with_columns(
    [
        pl.col("timestamp").dt.hour().alias("hour"),
        (pl.col("timestamp").dt.weekday() - 1).alias("dow"),
        pl.col("timestamp").dt.date().alias("date"),
        pl.col("timestamp").dt.truncate("1h").cast(pl.Datetime(TS_UNIT)).alias("weather_hour"),
        pl.col("segment_key").str.slice(0, pl.col("segment_key").str.len_chars() - 3).alias("segment_id"),
    ]
).with_columns(
    [
        (pl.col("dow") >= 5).alias("is_weekend"),
        ((pl.col("hour").cast(pl.Int32) * 100) + (pl.col("timestamp").dt.minute() // 30) * 30).alias("time_slot"),
    ]
)

print(df.select(["segment_key", "segment_id", "timestamp", "hour", "dow", "is_weekend", "time_slot", "date"]).head())

In [ ]:
# ==================================================================
# 4. is_bottleneck_slot 조인 (exact, segment_key + time_slot)
# ==================================================================
# ✅ v3: Train 기간만으로 재계산한 버전 사용 (leakage 제거)

bottleneck = pl.read_csv(BOTTLENECK_PATH).select(["segment_key", "time_slot", "is_bottleneck_slot"])

before = df.height
df = df.join(bottleneck, on=["segment_key", "time_slot"], how="left")
print(f"조인 후 shape: {df.shape} (조인 전 {before}행 유지되어야 정상)")
print(f"is_bottleneck_slot 결측률: {df['is_bottleneck_slot'].null_count() / df.height:.4%}")

In [ ]:
# ==================================================================
# 5. network_features 조인 (정적, segment_key 기준)
# ==================================================================

network = pl.read_parquet(NETWORK_PATH)
df = df.join(network, on="segment_key", how="left")
print(f"조인 후 shape: {df.shape}")
print(f"betweenness_pre 결측률: {df['betweenness_pre'].null_count() / df.height:.4%}")

In [ ]:
# ==================================================================
# 6. construction_lane_ratio_daily 조인 (segment_id + date)
# ==================================================================

lane_ratio = pl.read_parquet(LANE_RATIO_PATH).with_columns(pl.col("date").dt.date().alias("date"))

df = df.join(lane_ratio, on=["segment_id", "date"], how="left").with_columns(
    pl.col("lane_remain_ratio").fill_null(1.0)
)
print(f"조인 후 shape: {df.shape}")
print(f"lane_remain_ratio 결측(채움 후 0이어야 정상): {df['lane_remain_ratio'].null_count()}")

In [ ]:
# ==================================================================
# 7. incident_flag_10min 조인 (segment_key + timestamp)
# ==================================================================

incident = pl.read_parquet(INCIDENT_PATH).with_columns(pl.col("timestamp").cast(pl.Datetime(TS_UNIT)))

df = df.join(incident, on=["segment_key", "timestamp"], how="left").with_columns(
    [
        pl.col("incident_flag").fill_null(False),
        pl.col("incident_count").fill_null(0),
    ]
)
print(f"조인 후 shape: {df.shape}")
print(f"incident_flag=True 행 수: {df['incident_flag'].sum()}")

In [ ]:
# ==================================================================
# 8. weather_features 조인 (1시간 단위, 시 전체 공통값)
# ==================================================================
# ⚠ v3에서도 결측 528건(경계) 남음 - is_weather_alert/is_freezing 사용 시
# fillna(False) 필요 (xgb_model.ipynb에서 이미 처리 중).

weather = pl.read_parquet(WEATHER_PATH).with_columns(
    pl.col("timestamp").cast(pl.Datetime(TS_UNIT))
).rename({"timestamp": "weather_hour"})

df = df.join(weather, on="weather_hour", how="left")
print(f"조인 후 shape: {df.shape}")
print(f"precipitation_mm 결측률: {df['precipitation_mm'].null_count() / df.height:.4%}")

In [ ]:
# ==================================================================
# 9. prophet_features 조인 (segment_key + timestamp)
# ==================================================================

if PROPHET_PATH.exists():
    prophet = pl.read_parquet(PROPHET_PATH).with_columns(
        pl.col("timestamp").cast(pl.Datetime(TS_UNIT))
    ).rename({"split": "prophet_split"})

    df = df.join(prophet, on=["segment_key", "timestamp"], how="left")
    print(f"조인 후 shape: {df.shape}")
    print(f"y_hat_t30 결측 비율: {df['y_hat_t30'].null_count() / df.height:.1%} (기대치: 약 20%, 18/90 구간 미커버분)")
else:
    print(f"⚠ {PROPHET_PATH} 없음 - prophet_features.ipynb를 먼저 실행하세요. 이번엔 prophet 컬럼 없이 진행합니다.")

In [ ]:
# ==================================================================
# 10. 라벨(y) 생성: t+30분 as-of nearest(±10분 이내)
# ==================================================================

target = speed_sorted.select(
    [
        pl.col("segment_key"),
        pl.col("timestamp").alias("target_ts"),
        pl.col("V_segment").alias("target_speed"),
    ]
)

df = df.with_columns((pl.col("timestamp") + pl.duration(minutes=30)).alias("target_ts"))

df_sorted = df.sort(["segment_key", "target_ts"])
target_sorted = target.sort(["segment_key", "target_ts"])

df = df_sorted.join_asof(
    target_sorted,
    on="target_ts",
    by="segment_key",
    strategy="nearest",
    tolerance=LABEL_TOLERANCE,
)
print(f"target_speed 결측률(라벨 매칭 실패): {df['target_speed'].null_count() / df.height:.2%}")

n_before = df.height
df = df.filter(pl.col("V_segment").is_not_null() & pl.col("target_speed").is_not_null())
print(f"최종 유효 행: {df.height} / {n_before} ({df.height/n_before:.1%} 유지)")

df = df.with_columns(
    pl.when(pl.col("target_speed") >= 20)
    .then(0)
    .when(pl.col("target_speed") >= 15)
    .then(1)
    .otherwise(2)
    .alias("label")
)
print(df["label"].value_counts().sort("label"))

In [ ]:
# ==================================================================
# 11. split(Train/Val/Test) 부여
# ==================================================================

df = df.with_columns(
    pl.when(pl.col("timestamp") < pl.lit(TRAIN_END).str.to_datetime())
    .then(pl.lit("Train"))
    .when(pl.col("timestamp") < pl.lit(VAL_END).str.to_datetime())
    .then(pl.lit("Val"))
    .otherwise(pl.lit("Test"))
    .alias("split")
)

print(df.group_by("split").agg(pl.len()).sort("split"))
print()
print("split별 라벨 분포:")
print(df.group_by(["split", "label"]).agg(pl.len()).sort(["split", "label"]))

In [ ]:
# ==================================================================
# 12. 최종 컬럼 정리 + 저장 (v3)
# ==================================================================

final_cols = [
    "segment_key", "timestamp", "split",
    "V_segment", "speed_last_10min", "speed_ma_30min", "speed_ma_1h", "speed_change_rate",
    "hour", "dow", "is_weekend", "is_bottleneck_slot",
    "betweenness_pre", "betweenness_during", "road_rank", "lanes",
    "lane_remain_ratio",
    "incident_flag", "incident_count",
    "precipitation_mm", "is_weather_alert", "is_freezing",
    "target_speed", "label",
]
if "y_hat_t30" in df.columns:
    final_cols += ["y_hat_t30", "y_hat_lower_t30", "y_hat_upper_t30", "prophet_split"]

final_df = df.select(final_cols)
final_df.write_parquet(OUTPUT_PATH)

print(f"저장 완료: {OUTPUT_PATH.resolve()}")
print(f"최종 shape: {final_df.shape}")
print(final_df.schema)

In [ ]:
# ==================================================================
# 13. 최종 검증
# ==================================================================

null_summary = final_df.null_count().transpose(include_header=True, header_name="column", column_names=["null_count"])
null_summary = null_summary.with_columns((pl.col("null_count") / final_df.height * 100).round(2).alias("null_pct"))
print(null_summary.sort("null_pct", descending=True))

print()
print("=== 요약 ===")
print(f"총 행 수: {final_df.height:,}")
print(f"segment_key 수: {final_df['segment_key'].n_unique()}")
print(f"기간: {final_df['timestamp'].min()} ~ {final_df['timestamp'].max()}")
print()
print("※ v2 대비 변경점: is_bottleneck_slot을 Train 기간(<2026-06-14)만으로")
print("  재계산해서 leakage를 제거함 (v2 대비 판정이 달라진 슬롯은 0.2%뿐).")